In [23]:
# Imports
from pathlib import Path
from typing import List, Dict, Any
import os
import json
import pickle
import numpy as np

from pinecone import Pinecone, ServerlessSpec

from sklearn.feature_extraction.text import TfidfVectorizer

from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_google_genai import ChatGoogleGenerativeAI

from langchain_community.document_loaders import PyPDFLoader, Docx2txtLoader, TextLoader
from bs4 import BeautifulSoup

# Config

In [3]:
import os
from dotenv import load_dotenv

load_dotenv()

gemini_api_key = os.getenv("GEMINI_API_KEY")
pinecone_api_key = os.getenv("PINECONE_API_KEY")

In [5]:
## Define an Index name
RESUME_INDEX_NAME = "resume-hybrid-index"

#yf-idf file
TFIDF_PATH = "tfidf_vectorizer.pkl"

#Alpha value for hybrid search

# top-k values

In [20]:
from sentence_transformers import SentenceTransformer, CrossEncoder

#Dense Model
DENSE_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
dense_model = SentenceTransformer(DENSE_MODEL_NAME)


# Reranking defines
RERANK_MODEL_NAME = "cross-encoder/ms-marco-MiniLM-L-6-v2"
rerank_model = CrossEncoder(RERANK_MODEL_NAME)

In [6]:
# Getting pinecone Index
pc = Pinecone(api_key=pinecone_api_key)

index = pc.Index(RESUME_INDEX_NAME)
index

In [8]:
# LLM for MqE 
llm = ChatGoogleGenerativeAI(
    model = "gemini-2.5-flash",
    api_key = gemini_api_key,
)

# Retirval

In [13]:
def multi_query_ext(user_query):
    prompt = f"""
    Imagine you are helping retiver the most sutable canidates resume and you will be receving sample JD or user query about resume

    Write 3 variations based on the user question focusing different of reterival.. Do not explain what you are doing or the process

    Return only the below points
    1. The output should be sematically related to the user question
    2. Use different works compared to question but they should be related 
    3. Cover different aspects for resume filtering
    
    query_id:
    {user_query}
    """
    resp = llm.invoke(prompt)
    content = getattr(resp, "content", resp)
    return content

In [14]:
user_query = "Resume about data science"

In [16]:
mqe_query = multi_query_ext(user_query)
mqe_query

'1.  Profiles demonstrating expertise in machine learning, statistical inference, and predictive modeling.\n2.  Curricula vitae detailing practical experience with Python, R, SQL, and big data platforms for advanced analytics.\n3.  Documents highlighting quantitative analysis, experimental design, and the derivation of actionable insights from complex datasets.'

In [ ]:
final_query = user_query + '\n' + multi_query_ext(user_query)
final_query

In [21]:
# Hydbrid Retrival
def hybrid_query(query_text: str, alpha: float = 0.5, top_k: int = 8):
    """
    Hybrid search combining dense (1 - alpha) and sparse (alpha) scores.

    alpha = 0.0 => dense-only
    alpha = 1.0 => sparse-only (keyword)
    """
    alpha = float(alpha)
    alpha = max(0.0, min(1.0, alpha))

    # Load TF-IDF vectorizer trained during ingest
    with open(TFIDF_PATH, "rb") as f:
        vectorizer = pickle.load(f)

    # Dense query embedding
    q_dense = dense_model.encode([query_text], normalize_embeddings=True)[0]
    q_dense = (np.asarray(q_dense, dtype=float) * (1.0 - alpha)).tolist()

    # Sparse query vector
    q_sparse_csr = vectorizer.transform([query_text]).tocoo()
    if q_sparse_csr.nnz == 0:
        q_sparse = {"indices": [0], "values": [0.0]}
    else:
        q_sparse = {
            "indices": q_sparse_csr.col.tolist(),
            "values": (q_sparse_csr.data.astype(float) * alpha).tolist(),
        }

    # Query Pinecone
    res = index.query(
        vector=q_dense,
        sparse_vector=q_sparse,
        top_k=top_k,
        include_metadata=True,
    )

    out = []
    for m in res.get("matches", []):
        md = m.get("metadata", {}) or {}
        text = md.get("text", "") or ""
        preview = " ".join(text.split()[:120])  # short snippet

        out.append(
            {
                "id": m["id"],                 # resume_id
                "score": float(m["score"]),    # hybrid score
                "preview": preview,
                "metadata": md,
            }
        )
    return out

In [26]:
results = hybrid_query(mqe_query)
results

[{'id': 'Ashley_Phillips_Resume_32',
  'score': 0.291654289,
  'preview': 'Summary: Experienced accounting professional with 3+ years of progressive experience in financial reporting, analysis, and compliance. Proven track record of improving processes and delivering accurate financial information. Strong expertise in accounting principles and software applications. No specific projects were detailed in the resume. Skills: Variance Analysis, Trend Analysis, Portfolio Management, KPI Development, Data Analysis, Forecasting, Due Diligence, Financial Modeling, Microsoft Excel, Database Management, NetSuite, Financial Software, Python, SAP',
  'metadata': {'companies': ['Professional Accounting Partners'],
   'filename': 'Ashley_Phillips_Resume_32.pdf',
   'resume_id': 'Ashley_Phillips_Resume_32',
   'roles': ['Compliance Manager'],
   'skills': ['Data Analysis',
    'Database Management',
    'Due Diligence',
    'Financial Modeling',
    'Financial Software',
    'Forecasting',
    'KPI 

In [36]:
# Re-ranker
def reranker_crossencoder(query, results):
    pairs = [(query, r['preview']) for r in results]
    scores = rerank_model.predict(pairs)
    #scores = np.mod(scores)
    rescored = []
    for r, s in zip(results,scores):
        r2 = dict(r)
        r2['rerank_cross_score'] = float(s)
        rescored.append(r2)
    return sorted(rescored, key = lambda x:x['rerank_cross_score'], reverse= True)

In [38]:
rerank_results = reranker_crossencoder(mqe_query, results)
rerank_results

[{'id': 'Ashley_Phillips_Resume_32',
  'score': 0.291654289,
  'preview': 'Summary: Experienced accounting professional with 3+ years of progressive experience in financial reporting, analysis, and compliance. Proven track record of improving processes and delivering accurate financial information. Strong expertise in accounting principles and software applications. No specific projects were detailed in the resume. Skills: Variance Analysis, Trend Analysis, Portfolio Management, KPI Development, Data Analysis, Forecasting, Due Diligence, Financial Modeling, Microsoft Excel, Database Management, NetSuite, Financial Software, Python, SAP',
  'metadata': {'companies': ['Professional Accounting Partners'],
   'filename': 'Ashley_Phillips_Resume_32.pdf',
   'resume_id': 'Ashley_Phillips_Resume_32',
   'roles': ['Compliance Manager'],
   'skills': ['Data Analysis',
    'Database Management',
    'Due Diligence',
    'Financial Modeling',
    'Financial Software',
    'Forecasting',
    'KPI 